# Part 4-B — RAG Implementation

**Components:**
- Vector DB: FAISS (`faiss-cpu`) — pure pip install, no server required
- Embedding model: `paraphrase-multilingual-MiniLM-L12-v2` (English + Turkish)
- LLM: Qwen 2.5 7B Instruct (4-bit quantized)

**Steps:**
1. Build FAISS index from WMT16 train corpus
2. Demo retrieval for sample queries
3. Run RAG translation on 200 test samples
4. Compute COMET score

> Index is cached to `data/faiss_index.*` — subsequent runs load instantly.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import json
import numpy as np
import matplotlib.pyplot as plt

from modules.dataset import load_wmt16, get_samples, get_corpus
from modules.model import load_model
from modules.retrieval import (
    load_embedding_model,
    build_faiss_index,
    retrieve_examples,
    make_retriever,
)
from modules.translation import translate_with_rag, load_cached_translations
from modules.evaluation import load_comet_model, compute_comet

os.makedirs('../data', exist_ok=True)
os.makedirs('../results', exist_ok=True)

## 4-B.1 Load Corpus & Test Data

In [ ]:
# Load train corpus for indexing (cap at 50K for speed; full 207K also works)
CORPUS_MAX = 50_000

print("Loading WMT16 train split for RAG corpus...")
train_ds = load_wmt16(split='train')
corpus_pairs = get_corpus(train_ds, max_size=CORPUS_MAX)
print(f"Corpus: {len(corpus_pairs)} pairs")

In [ ]:
# Load test data (same 200 samples as Part 3 for comparability)
test_ds = load_wmt16(split='test')
en_sentences, tr_references = get_samples(test_ds, n=200, seed=42)
print(f"Test samples: {len(en_sentences)}")

## 4-B.2 Embedding Generation & FAISS Index

In [ ]:
# Load embedding model
emb_model = load_embedding_model()

In [ ]:
# Build or load cached FAISS index
INDEX_PATH = '../data/faiss_index'

index, corpus_pairs = build_faiss_index(
    corpus_pairs,
    emb_model,
    index_path=INDEX_PATH
)

print(f"\nFAISS index stats:")
print(f"  Total vectors : {index.ntotal:,}")
print(f"  Dimension     : {index.d}")
print(f"  Index type    : {type(index).__name__} (exact cosine search)")

## 4-B.3 Similarity Search Demo

In [ ]:
# Demo: retrieve top-5 examples for 3 queries
demo_queries = [
    "The parliament approved the new budget plan.",
    "Scientists discovered a new species in the Amazon rainforest.",
    "The company reported record profits in the third quarter.",
]

for query in demo_queries:
    examples = retrieve_examples(query, index, corpus_pairs, emb_model, k=3)
    print(f"Query: {query}")
    print("Top-3 retrieved examples:")
    for i, (en, tr) in enumerate(examples, 1):
        en_disp = en[:70] + '...' if len(en) > 70 else en
        tr_disp = tr[:70] + '...' if len(tr) > 70 else tr
        print(f"  [{i}] EN: {en_disp}")
        print(f"       TR: {tr_disp}")
    print()

## 4-B.4 Integration with LLM Pipeline

In [ ]:
# Show a full RAG prompt for one example
from modules.prompts import rag_prompt

sample_query = en_sentences[0]
examples = retrieve_examples(sample_query, index, corpus_pairs, emb_model, k=5)
prompt = rag_prompt(sample_query, examples, direction='en->tr')

print("=" * 60)
print("FULL RAG PROMPT (sample):")
print("=" * 60)
print(prompt)
print("=" * 60)

In [ ]:
# Load LLM
model, tokenizer = load_model(quantize=True)

## 4-B.5 RAG Translation — 200 Test Samples

In [ ]:
RAG_PATH = '../results/rag_translations.json'

# Create retriever closure (k=5 examples per query)
retriever = make_retriever(index, corpus_pairs, emb_model, k=5)

if os.path.exists(RAG_PATH):
    print(f"Loading cached RAG translations from {RAG_PATH}")
    rag_translations = load_cached_translations(RAG_PATH)
else:
    rag_translations = translate_with_rag(
        model, tokenizer, en_sentences,
        retriever=retriever,
        direction='en->tr',
        save_path=RAG_PATH
    )

print(f"\nSample RAG outputs:")
for i in range(3):
    print(f"  EN : {en_sentences[i]}")
    print(f"  REF: {tr_references[i]}")
    print(f"  RAG: {rag_translations[i]}")
    print()

## 4-B.6 COMET Evaluation

In [ ]:
comet_model = load_comet_model()

print("Computing COMET — RAG (5-shot)...")
rag_result = compute_comet(en_sentences, rag_translations, tr_references, comet_model)

rag_scores = {'rag_5shot': rag_result['system_score']}

# Save for Part 5
with open('../results/part4_comet_scores.json', 'w') as f:
    json.dump(rag_scores, f, indent=2)

print(f"\nRAG 5-shot COMET: {rag_result['system_score']:.4f}")

## 4-B.7 Retrieval Quality Analysis

In [ ]:
import faiss

# Compute similarity scores for the first 50 test queries
from modules.retrieval import embed_sentences

query_embs = embed_sentences(en_sentences[:50], emb_model, show_progress=False)
scores, _ = index.search(query_embs, 5)
top1_scores = scores[:, 0]  # highest similarity per query

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(top1_scores, bins=20, color='#2ecc71', edgecolor='white', alpha=0.9)
ax.set_xlabel('Top-1 Cosine Similarity Score', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Distribution of Top-1 Retrieval Similarity Scores\n(first 50 test queries)', fontsize=12)
ax.axvline(top1_scores.mean(), color='red', linestyle='--',
           label=f'Mean: {top1_scores.mean():.3f}')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/retrieval_similarity.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Mean top-1 similarity: {top1_scores.mean():.3f}")
print(f"Min: {top1_scores.min():.3f}  Max: {top1_scores.max():.3f}")